# CellPatcher strategy examples

This notebook demonstrates CellPatcher patching strategies using **small synthetic NumPy arrays**.

- It uses `run_patching`, the main orchestration function for in-memory arrays.
- It is **not** the full app UI notebook.
- It does **not** require image files or external datasets.
- It does **not** perform DINO/Torch inference.


In [ ]:
# @title Imports and synthetic data helpers
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

from cellpatcher import (
    CellPatcherConfig,
    CommonConfig,
    CellConfig,
    RandomConfig,
    GridConfig,
    ExtractionConfig,
    ExportConfig,
    run_patching,
    scan_cell_patch_size,
)


def make_synthetic_image_and_labels():
    image = np.zeros((96, 96), dtype=np.float32)
    yy, xx = np.ogrid[:96, :96]

    # Interior object 1 (circle)
    c1 = (yy - 30) ** 2 + (xx - 28) ** 2 <= 10 ** 2
    image[c1] = 1.0

    # Interior object 2 (rectangle)
    image[58:78, 60:82] = 0.8

    # Border-touching object (left edge)
    image[10:26, 0:12] = 0.7

    # Tiny object
    image[84:86, 84:86] = 1.2

    # Mild background variation
    image += np.linspace(0.0, 0.1, image.size, dtype=np.float32).reshape(image.shape)

    labels = np.zeros((96, 96), dtype=np.int32)
    labels[c1] = 1
    labels[58:78, 60:82] = 2
    labels[10:26, 0:12] = 3
    labels[84:86, 84:86] = 4

    return image, labels


def show_image_and_labels(image, labels):
    fig, axs = plt.subplots(1, 2, figsize=(8, 4))
    axs[0].imshow(image, cmap="gray")
    axs[0].set_title("Synthetic image")
    axs[0].axis("off")

    axs[1].imshow(labels, cmap="nipy_spectral", interpolation="nearest")
    axs[1].set_title("Synthetic labels")
    axs[1].axis("off")
    plt.tight_layout()
    plt.show()


def show_patch_gallery(patches, n=8, title="Patches"):
    if not patches:
        print("No patches to display.")
        return

    n_show = min(n, len(patches))
    fig, axs = plt.subplots(1, n_show, figsize=(2.7 * n_show, 2.7))
    if n_show == 1:
        axs = [axs]

    for i in range(n_show):
        arr = np.asarray(patches[i])
        if arr.ndim > 2:
            arr = np.take(arr, 0, axis=0)
        axs[i].imshow(arr, cmap="gray")
        axs[i].set_title(f"Patch {i}")
        axs[i].axis("off")

    fig.suptitle(title)
    plt.tight_layout()
    plt.show()


## Synthetic data used in all examples

All strategy examples below use the same synthetic image and label mask.

- Labels are used for **cell mode** and optionally as a sampling mask for **random mode**.
- Axes are set to `YX`.


In [ ]:
# @title Create synthetic example data (or optionally use bundled real TIFF fixture)
USE_REAL_TIFF_FIXTURE = False

if USE_REAL_TIFF_FIXTURE:
    repo_root = Path.cwd()
    fixture_image = repo_root / 'tests' / 'fixtures' / 'cellpatcher_demo_image.tif'
    fixture_labels = repo_root / 'tests' / 'fixtures' / 'cellpatcher_demo_labels.tif'
    if fixture_image.exists() and fixture_labels.exists():
        image = read_array(fixture_image)
        labels = read_array(fixture_labels)
        print('Using real TIFF fixture')
        print(' image shape/dtype:', image.shape, image.dtype)
        print(' labels shape/dtype:', labels.shape, labels.dtype)
    else:
        print('WARNING: fixture files not found, falling back to synthetic example data')
        image, labels = make_synthetic_image_and_labels()
else:
    image, labels = make_synthetic_image_and_labels()

print('image shape/dtype:', image.shape, image.dtype)
print('labels shape/dtype:', labels.shape, labels.dtype)
print('unique labels:', np.unique(labels))

show_image_and_labels(image, labels)


## Cell mode

Cell mode creates one patch per segmented object/cell.

It supports labeled masks and binary masks. Key parameters include:

- `patch_size`
- `exclude_border`
- `center_on`
- `min_area`
- `max_area`
- `binary_mask_policy`
- `apply_mask`

Use `scan_cell_patch_size` to estimate dataset-level patch sizes.
When `apply_mask=True`, pixels outside the selected cell label are cleared before resize/export.


In [ ]:
# @title Cell mode: one patch per labeled object
summary = scan_cell_patch_size(
    image_shapes=[image.shape],
    labels_list=[labels],
    config=CellConfig(exclude_border=True, min_area=5),
    image_axes="YX",
    mask_axes="YX",
)

patch_size = summary.recommended_size("max")

config = CellPatcherConfig(mode="cell")
config.common = CommonConfig(image_axes="YX", mask_axes="YX", extract_arrays=True)
config.cell = CellConfig(
    patch_size=patch_size,
    exclude_border=True,
    min_area=5,
    center_on="bbox_center",
)
config.extraction = ExtractionConfig(
    apply_mask=True,
    mask_fill_value=0,
    output_size=32,
)

result = run_patching(images=image, masks=labels, config=config)

print("cell patches:", len(result.patches))
print("recommended patch_size:", patch_size)
for r in result.patches[:3]:
    print(r.to_dict())

show_patch_gallery(result.patch_arrays, n=8, title="Cell mode patches")


## Random mode

Random mode samples fixed-size patches at random locations.

Key parameters:
- `patch_size`
- `n_patches`
- `seed`
- optional mask constraint

Sampling is with replacement, so duplicate locations are possible.
Useful for background/control patches or random coverage.


In [ ]:
# @title Random mode: seeded random patches
sampling_mask = labels > 0

config = CellPatcherConfig(mode="random")
config.common = CommonConfig(image_axes="YX", mask_axes="YX", extract_arrays=True)
config.random = RandomConfig(
    patch_size=24,
    n_patches=6,
    seed=42,
)
config.extraction = ExtractionConfig(output_size=32)

result = run_patching(images=image, masks=sampling_mask, config=config)

print("random patches:", len(result.patches))
for r in result.patches[:3]:
    print(r.to_dict())

show_patch_gallery(result.patch_arrays, n=6, title="Random mode patches")


## Grid mode

Grid mode divides the image into regular tiles.

Key parameters:
- `tile_size` — side length of each square tile in pixels
- `overlap` — fractional overlap between adjacent tiles (0.0 = no overlap, must be < 1.0)
- `include_partial` — when `True`, tiles at the image border that are smaller than `tile_size` are included; when `False` they are discarded

Useful for tiling a full field of view into model-ready patches. No mask is required.


In [ ]:
# @title Grid mode: regular tiling with overlap
config = CellPatcherConfig(mode="grid")
config.common = CommonConfig(image_axes="YX", extract_arrays=True)
config.grid = GridConfig(
    tile_size=32,
    overlap=0.25,
    include_partial=True,
)
config.extraction = ExtractionConfig(output_size=32)

result = run_patching(images=image, config=config)

print("grid patches:", len(result.patches))
for r in result.patches[:3]:
    print(r.to_dict())

show_patch_gallery(result.patch_arrays, n=8, title="Grid mode patches")


## Whole-FOV mode

Whole-FOV mode returns one patch covering the entire field of view.

Useful when downstream processing expects a single whole-image patch.
You can still combine it with extraction, resizing, and export settings.


In [ ]:
# @title Whole-FOV mode: one patch for the full image
config = CellPatcherConfig(mode="whole_fov")
config.common = CommonConfig(image_axes="YX", extract_arrays=True)
config.extraction = ExtractionConfig(output_size=64)

result = run_patching(images=image, config=config)

print("whole_fov patches:", len(result.patches))
print(result.patches[0].to_dict())

show_patch_gallery(result.patch_arrays, n=1, title="Whole-FOV patch")


## Optional filtering/export note

These examples keep extracted arrays in memory via `extract_arrays=True`
(returned as `result.patch_arrays`).

Enabling export writes patch files to disk **independently** of `extract_arrays`.
To write files without retaining arrays in memory, set `extract_arrays=False` and
`export.enabled=True`. Both can also be `True` simultaneously.

For export, set:
- `config.common.output_dir` — destination directory (string path)
- `config.export.enabled = True`
- `config.export.image_format = "npy"` or `"tif"`
- `config.export.save_metadata_csv = True`

Intensity-based filtering can be added through `FilterConfig` (see `config.filters`).


In [ ]:
# @title Optional export example
RUN_EXPORT_EXAMPLE = False

if RUN_EXPORT_EXAMPLE:
    output_dir = "cellpatcher_example_output"
    config = CellPatcherConfig(mode="grid")
    config.common = CommonConfig(
        image_axes="YX",
        output_dir=output_dir,
        extract_arrays=False,
    )
    config.grid = GridConfig(tile_size=32, overlap=0.0, include_partial=True)
    config.export = ExportConfig(
        enabled=True,
        image_format="npy",
        save_metadata_csv=True,
    )
    result = run_patching(images=image, config=config)
    print(result.exported_files)
    print(result.metadata_file)
else:
    print("Set RUN_EXPORT_EXAMPLE = True to write example patches to disk.")
